<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/notebook-02-tokenizer-transition/notebooks/02_tokenizer_training_and_corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 02 — Tokenizer Training & Corpus Construction

This notebook begins from the verified data contract established in **Notebook 01 — Data Preparation & Corpus Audit**. It independently reloads the immutable WikiText-103 revision, applies the same locked normalization and article-reconstruction logic, trains the project tokenizer from scratch, validates it, records its checksum, and constructs the exact 20,000,000-token model-training corpus.

Notebook 01 is a completed audit artifact. Notebook 02 must not depend on Notebook 01's in-memory state. Reusable preprocessing logic should live in project source code and be imported here.

### Pipeline boundary

```text
Notebook 01: raw WikiText -> verified normalized articles
Notebook 02: normalized articles -> tokenizer -> exact 20M-token corpus
Notebook 03: tokenizer/corpus -> Transformer architecture
```

### Locked inputs and constraints

- Dataset: `Salesforce/wikitext`, `wikitext-103-raw-v1`
- Immutable Hub revision: `b08601e04326c79dfdd32d625aee71d232d685c3`
- Tokenizer: byte-level BPE trained from scratch
- Vocabulary size: 16,384 total tokens, including registered special tokens
- Tokenizer-training text: full normalized official training split only
- Model-training corpus: exactly 20,000,000 tokenizer-produced tokens
- Sampling seed: 42
- Validation remains development-visible; test remains untouched until final evaluation
- Article boundaries and normalization must reproduce Notebook 01's verified 28,472 training documents and 60 validation documents before tokenizer work proceeds

Canonical references: `docs/PROJECT_CONTEXT.md` and `docs/DECISION_REGISTER.md`.

## 1. Scope of the first Notebook 02 chunk

Before training the tokenizer, this notebook will first establish a clean reproducible handoff from Notebook 01:

1. import the shared data-preparation helpers;
2. reload the pinned WikiText-103 revision;
3. reproduce the locked normalization and article reconstruction;
4. hard-assert the verified document counts; and
5. only then define the tokenizer's document-boundary/EOS token.

Tokenizer training, validation, checksumming, and 20M-token corpus construction belong to later reviewable chunks in this notebook.

### Pause here

Notebook 02 has been created as a new artifact. The next implementation chunk is to extract the locked reusable WikiText normalization/article-reconstruction logic into `src/data.py` and verify that Notebook 02 reproduces Notebook 01's audited counts without relying on notebook state.